# Prediccion por **centro comercial**: ingresos por acceso y hora

Este cuaderno entrena un modelo global y permite generar el **pronostico de 24 horas** para una **fecha especi­fica** y un **centro comercial especifico** (`id_cc`).

## Flujo
1. Cargar y limpiar datos
2. Features (calendario + lags/rollings)
3. Split temporal y entrenamiento (modelo global)
4. MÃƒÆ’Ã‚Â©tricas (global y por acceso)
5. **Pronostico para un `center_id` y `target_date`** Tabla por hora y acceso

**Datos requeridos**: columnas `timestamp`, `acceso_id`, `id_cc`, `ins` (ingresos por hora).

In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from xgboost.core import XGBoostError
import matplotlib.pyplot as plt

pd.options.display.max_columns = 120
pd.options.display.width = 160

DATA_PATH = Path(r'C:\\Users\\wilte\\OneDrive\\Escritorio\\Ciencias de datos\\Mid Term')
CSV_FILE = DATA_PATH / 'Datos_anonimizados.csv'  # <-- actualiza a tu ruta

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred, eps: float = 1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def to_float32(matrix):
    return np.asarray(matrix, dtype=np.float32)

print('Ready!')

Ready!


In [28]:
# Cargar datos: si no existe tu CSV, se genera uno de ejemplo
if CSV_FILE.exists():
    df = pd.read_csv(CSV_FILE)
else:
    rng = pd.date_range('2025-05-01', periods=90*24, freq='H')
    centers = list(range(1, 9))  # 8 centros
    rows = []
    for cid in centers:
        accesos = [f'CC{cid}_{i}' for i in range(1, 6)]  # 5 accesos por centro (ejemplo)
        for acc in accesos:
            base = np.maximum(0, 60 + 15*np.sin(np.arange(len(rng))/24*2*np.pi) + np.random.normal(0, 10, len(rng)))
            rows.append(pd.DataFrame({
                'timestamp': rng,
                'acceso_id': acc,
                'id_cc': cid,
                'ins': base.astype(int)
            }))
    df = pd.concat(rows, ignore_index=True)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['acceso_id','timestamp']).reset_index(drop=True)
df.head()

,registro_id,id_cc,timestamp,acceso_id,nombre_comercial_acceso,ins
0,2527527,1,2024-10-01 08:00:00,CC1_1,entrada_1,2
1,2527584,1,2024-10-01 09:00:00,CC1_1,entrada_1,140
2,2527659,1,2024-10-01 10:00:00,CC1_1,entrada_1,528
3,2527735,1,2024-10-01 11:00:00,CC1_1,entrada_1,827
4,2527825,1,2024-10-01 12:00:00,CC1_1,entrada_1,902


In [29]:
# Limpieza: duplicates + garantizar frecuencia horaria por acceso
df = df.drop_duplicates(['acceso_id','timestamp'])

def ensure_hourly(d):
    idx = pd.date_range(d['timestamp'].min(), d['timestamp'].max(), freq='h')
    d = d.set_index('timestamp').reindex(idx)
    d.index.name = 'timestamp'
    d['acceso_id'] = d['acceso_id'].ffill().bfill()
    d['id_cc'] = d['id_cc'].ffill().bfill()
    d['ins'] = d['ins'].fillna(0)
    return d.reset_index()

df = df.groupby('acceso_id', group_keys=False).apply(ensure_hourly)
df = df.sort_values(['acceso_id','timestamp']).reset_index(drop=True)
print('Rango temporal:', df['timestamp'].min(), '->', df['timestamp'].max())

Rango temporal: 2024-10-01 08:00:00 -> 2025-09-30 22:00:00


C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\533001828.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('acceso_id', group_keys=False).apply(ensure_hourly)


In [30]:
# Features (calendario + lags/rollings)
def add_calendar_features(d):
    out = d.copy()
    out['hour'] = out['timestamp'].dt.hour
    out['dow'] = out['timestamp'].dt.dayofweek
    out['dom'] = out['timestamp'].dt.day
    out['week'] = out['timestamp'].dt.isocalendar().week.astype(int)
    out['month'] = out['timestamp'].dt.month
    out['is_weekend'] = (out['dow']>=5).astype(int)
    return out

def add_lag_roll_features(d, lags=(1,24,168), roll_windows=(24,168)):
    out = d.sort_values(['acceso_id','timestamp']).copy()
    for lag in lags:
        out[f'lag_{lag}'] = out.groupby('acceso_id')['ins'].shift(lag)
    for w in roll_windows:
        out[f'roll_mean_{w}'] = (
            out.groupby('acceso_id')['ins'].transform(lambda s: s.shift(1).rolling(w, min_periods=max(2, w//4)).mean())
        )
        out[f'roll_std_{w}'] = (
            out.groupby('acceso_id')['ins'].transform(lambda s: s.shift(1).rolling(w, min_periods=max(2, w//4)).std())
        )
    return out

df_feat = add_calendar_features(df)
df_feat = add_lag_roll_features(df_feat)
df_feat.head(3)

,timestamp,registro_id,id_cc,acceso_id,nombre_comercial_acceso,ins,hour,dow,dom,week,month,is_weekend,lag_1,lag_24,lag_168,roll_mean_24,roll_std_24,roll_mean_168,roll_std_168
0,2024-10-01 08:00:00,2527527.0,1.0,CC1_1,entrada_1,2.0,8,1,1,40,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-01 09:00:00,2527584.0,1.0,CC1_1,entrada_1,140.0,9,1,1,40,10,0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-01 10:00:00,2527659.0,1.0,CC1_1,entrada_1,528.0,10,1,1,40,10,0,140.0,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Split temporal
horizon_days = 28
tmax = df_feat['timestamp'].max()
cutoff = tmax - pd.Timedelta(days=horizon_days)
train = df_feat[df_feat['timestamp'] <= cutoff].copy()
valid = df_feat[df_feat['timestamp'] > cutoff].copy()

feature_cols = [
    'hour','dow','dom','week','month','is_weekend',
    'lag_1','lag_24','lag_168','roll_mean_24','roll_std_24','roll_mean_168','roll_std_168',
    'acceso_id','id_cc'
]
target_col = 'ins'

train = train.dropna(subset=[c for c in feature_cols if c not in ['acceso_id','id_cc']])
valid = valid.dropna(subset=[c for c in feature_cols if c not in ['acceso_id','id_cc']])

X_train = train[feature_cols].copy(); y_train = train[target_col].astype(float).values
X_valid = valid[feature_cols].copy(); y_valid = valid[target_col].astype(float).values

cat_cols = ['acceso_id','id_cc']
num_cols = [c for c in feature_cols if c not in cat_cols]
pre = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols)
])

cast_to_float32 = FunctionTransformer(to_float32, validate=False)

xgb_params_gpu = dict(
    objective='reg:squarederror',
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    device='cuda',
    max_depth=8,
    learning_rate=0.08,
    n_estimators=600,
    subsample=0.9,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    random_state=42,
)
model_gpu = XGBRegressor(**xgb_params_gpu)

pipe = Pipeline([
    ('pre', pre),
    ('cast', cast_to_float32),
    ('xgb', model_gpu),
])

used_device = 'cuda'
try:
    pipe.fit(X_train, y_train)
except XGBoostError as ex:
    print('⚠️ Entrenamiento GPU falló, se utilizará CPU. Detalle:', ex)
    xgb_params_cpu = dict(xgb_params_gpu)
    xgb_params_cpu['device'] = 'cpu'
    xgb_params_cpu['tree_method'] = 'hist'
    xgb_params_cpu['predictor'] = 'auto'
    model_cpu = XGBRegressor(**xgb_params_cpu)
    pipe = Pipeline([
        ('pre', pre),
        ('cast', cast_to_float32),
        ('xgb', model_cpu),
    ])
    pipe.fit(X_train, y_train)
    used_device = 'cpu'

pred_valid = pipe.predict(X_valid)
print('Dispositivo utilizado:', used_device)
print('Valid RMSE:', rmse(y_valid, pred_valid))
print('Valid MAPE (%):', mape(y_valid, pred_valid))

⚠️ Entrenamiento GPU falló, se utilizará CPU. Detalle: Invalid Input: 'gpu_hist', valid values are: {'approx', 'auto', 'exact', 'hist'}


c:\Users\wilte\OneDrive\Escritorio\Ciencias de datos\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [22:12:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Dispositivo utilizado: cpu
Valid RMSE: 49.863938843723325
Valid MAPE (%): 115189021.10617962


## Nota sobre entrenamiento GPU

- Este notebook entrena usando XGBRegressor con device='cuda'. Asegúrate de contar con una GPU compatible y los drivers CUDA instalados.

- Si no hay GPU disponible, el bloque de entrenamiento realiza fallback automático a CPU y lo indicará en la salida.

- Tras ejecutar el entrenamiento, vuelve a correr la celda que guarda el pipeline para actualizar modelo_ingresos.pkl.

- La aplicación Flask (pp.py) necesita tener instaladas las dependencias: pip install xgboost lightgbm.

In [32]:
# Baseline semanal
vb = valid.copy()
vb['yhat_naive'] = vb['lag_168'].values
print('Baseline RMSE:', rmse(vb[target_col], vb['yhat_naive']))
print('Baseline MAPE (%):', mape(vb[target_col], vb['yhat_naive']))

Baseline RMSE: 78.18158965440874
Baseline MAPE (%): 209736286.8479719


In [33]:
# MÃƒÆ’Ã‚Â©tricas por acceso
df_eval = valid.copy(); df_eval['y_pred'] = pred_valid
per_acc = (df_eval.groupby('acceso_id')
                   .apply(lambda g: pd.Series({'RMSE': rmse(g[target_col], g['y_pred']), 'MAPE_%': mape(g[target_col], g['y_pred'])}))
                   .reset_index()
                   .sort_values('RMSE'))
per_acc.head(15)

C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\3590458764.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'RMSE': rmse(g[target_col], g['y_pred']), 'MAPE_%': mape(g[target_col], g['y_pred'])}))


,acceso_id,RMSE,MAPE_%
49,CC5_8,0.587334,4.832259e+07
50,CC5_9,0.587334,4.832259e+07
47,CC5_6,0.587334,4.832259e+07
48,CC5_7,0.587554,4.833294e+07
41,CC5_10,3.462911,1.785134e+08
53,CC6_3,5.934575,5.190454e+07
29,CC3_5,7.646486,2.216196e+07
38,CC4_6,7.848223,4.995684e+07
70,CC8_4,8.517993,4.811295e+07
30,CC3_6,8.778638,1.629938e+07


In [ ]:
# Pronostico 24h para *un centro* y fecha objetivo
def forecast_center_date(df_hist: pd.DataFrame, model_pipe: Pipeline, center_id: int, target_date: str) -> pd.DataFrame:
    """Pronostica 24h para los accesos de un centro en target_date.
    Devuelve columnas: ['id_cc','acceso_id','timestamp','yhat']
    """
    hist = df_hist.copy().sort_values(['acceso_id','timestamp']).reset_index(drop=True)
    # filtrar accesos del centro
    acc_center = hist.loc[hist['id_cc']==center_id, 'acceso_id'].unique()
    if len(acc_center)==0:
        raise ValueError(f'No hay accesos para id_cc={center_id}')

    date0 = pd.to_datetime(target_date)
    horizon = pd.date_range(date0, date0 + pd.Timedelta(hours=23), freq='H')

    def add_calendar_features(d):
        out = d.copy()
        out['hour'] = out['timestamp'].dt.hour
        out['dow'] = out['timestamp'].dt.dayofweek
        out['dom'] = out['timestamp'].dt.day
        out['week'] = out['timestamp'].dt.isocalendar().week.astype(int)
        out['month'] = out['timestamp'].dt.month
        out['is_weekend'] = (out['dow']>=5).astype(int)
        return out

    def add_lag_roll_features(d, lags=(1,24,168), roll_windows=(24,168)):
        out = d.sort_values(['acceso_id','timestamp']).copy()
        for lag in lags:
            out[f'lag_{lag}'] = out.groupby('acceso_id')['ins'].shift(lag)
        for w in roll_windows:
            out[f'roll_mean_{w}'] = out.groupby('acceso_id')['ins'].transform(lambda s: s.shift(1).rolling(w, min_periods=max(2, w//4)).mean())
            out[f'roll_std_{w}'] = out.groupby('acceso_id')['ins'].transform(lambda s: s.shift(1).rolling(w, min_periods=max(2, w//4)).std())
        return out

    preds = []
    for acc in acc_center:
        h = hist[hist['acceso_id']==acc].copy()
        h = h.set_index('timestamp').asfreq('H')
        h['acceso_id'] = h['acceso_id'].ffill(); h['id_cc'] = h['id_cc'].ffill()
        cur = h.copy()
        for ts in horizon:
            row = pd.DataFrame({'timestamp':[ts],'acceso_id':[acc],'id_cc':[center_id]})
            row = add_calendar_features(row)
            tmp = cur.reset_index().rename(columns={'index':'timestamp'})
            tmp = pd.concat([tmp, row], ignore_index=True)
            tmp = add_lag_roll_features(tmp)
            row_feat = tmp[tmp['timestamp']==ts].copy()
            for c in ['lag_1','lag_24','lag_168','roll_mean_24','roll_std_24','roll_mean_168','roll_std_168']:
                if row_feat[c].isna().any():
                    val = cur['ins'].tail(24).mean() if c.startswith(('lag_','roll_mean_')) else cur['ins'].tail(168).std()
                    row_feat[c] = row_feat[c].fillna(0.0 if not np.isfinite(val) else float(val))
            feat_cols = ['hour','dow','dom','week','month','is_weekend','lag_1','lag_24','lag_168','roll_mean_24','roll_std_24','roll_mean_168','roll_std_168','acceso_id','id_cc']
            yhat = float(model_pipe.predict(row_feat[feat_cols])[0])
            yhat = max(0.0, yhat)
            preds.append({'id_cc': center_id, 'acceso_id': acc, 'timestamp': ts, 'yhat': yhat})
            cur.loc[ts, 'ins'] = yhat

    out = pd.DataFrame(preds).sort_values(['acceso_id','timestamp']).reset_index(drop=True)
    return out

center_id = 1          # <-- id_cc del centro que deseas pronosticar
target_date = (df_feat['timestamp'].max() + pd.Timedelta(days=1)).date().isoformat()  # o '2025-10-20'
print('Centro:', center_id, '| Fecha objetivo:', target_date)
forecast_cc = forecast_center_date(df[['timestamp','acceso_id','id_cc','ins']], pipe, center_id, target_date)
forecast_cc.head(12)

Centro: 1 | Fecha objetivo: 2025-10-01


C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\153335696.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  horizon = pd.date_range(date0, date0 + pd.Timedelta(hours=23), freq='H')
C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\153335696.py:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  h = h.set_index('timestamp').asfreq('H')
C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\153335696.py:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  h = h.set_index('timestamp').asfreq('H')
C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\153335696.py:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  h = h.set_index('timestamp').asfreq('H')
C:\Users\wilte\AppData\Local\Temp\ipykernel_25540\153335696.py:37: FutureWarning: 'H' is deprecated and will be removed in a future vers

,id_cc,acceso_id,timestamp,yhat
0,1,CC1_1,2025-10-01 00:00:00,57.949196
1,1,CC1_1,2025-10-01 01:00:00,67.061241
2,1,CC1_1,2025-10-01 02:00:00,70.390579
3,1,CC1_1,2025-10-01 03:00:00,70.390579
4,1,CC1_1,2025-10-01 04:00:00,72.215652
5,1,CC1_1,2025-10-01 05:00:00,87.469315
6,1,CC1_1,2025-10-01 06:00:00,98.255394
7,1,CC1_1,2025-10-01 07:00:00,98.001556
8,1,CC1_1,2025-10-01 08:00:00,101.836411
9,1,CC1_1,2025-10-01 09:00:00,137.166016


In [35]:
# Tabla final: horas en filas, accesos en columnas, solo del centro seleccionado
pivot_cc = forecast_cc.pivot_table(index='timestamp', columns='acceso_id', values='yhat', aggfunc='sum').round(2)
pivot_cc.head(24)

acceso_id,CC1_1,CC1_10,CC1_11,CC1_12,CC1_13,CC1_14,CC1_15,CC1_16,CC1_2,CC1_3,CC1_4,CC1_5,CC1_6,CC1_7,CC1_8,CC1_9
timestamp,,,,,,,,,,,,,,,,
2025-10-01 00:00:00,57.95,0.82,33.20,5.45,3.99,5.35,0.87,25.50,0.00,3.82,8.59,5.46,1.54,7.06,11.53,1.96
2025-10-01 01:00:00,67.06,0.82,36.92,5.45,3.99,5.35,0.87,25.74,0.00,3.82,8.59,5.46,1.54,7.06,11.53,1.96
2025-10-01 02:00:00,70.39,0.82,40.25,5.45,3.99,5.35,0.87,26.52,0.00,3.82,8.59,5.46,1.54,7.06,11.53,1.96
2025-10-01 03:00:00,70.39,1.00,40.25,5.63,4.17,5.53,1.05,26.52,0.00,4.00,8.77,5.64,1.72,7.24,11.70,2.14
2025-10-01 04:00:00,72.22,1.00,41.54,7.78,5.03,6.39,1.05,28.48,0.00,4.86,10.51,6.50,2.12,8.10,13.45,2.54
2025-10-01 05:00:00,87.47,1.00,52.22,10.20,6.91,8.81,1.05,31.57,0.00,6.74,12.93,8.92,2.12,10.52,15.87,2.54
2025-10-01 06:00:00,98.26,1.31,54.77,10.31,7.08,8.97,1.41,31.59,0.31,6.78,13.10,8.96,2.37,10.63,16.04,2.84
2025-10-01 07:00:00,98.00,1.21,54.62,10.21,6.98,8.88,1.32,33.60,0.19,6.69,13.00,8.86,2.27,10.53,15.94,2.75
2025-10-01 08:00:00,101.84,1.38,54.64,10.32,7.05,9.06,1.56,33.21,0.48,6.91,17.22,9.09,2.50,10.76,16.12,2.93


In [36]:
import joblib

# Guarda el modelo y el preprocesador completo (pipeline)
joblib.dump(pipe, 'modelo_ingresos.pkl')
print("Modelo guardado en modelo_ingresos.pkl")


Modelo guardado en modelo_ingresos.pkl
